In [ ]:
from fpylll import IntegerMatrix, LLL, BKZ
from sage.all import matrix, ZZ

def flatter_reduce(L):
    A = IntegerMatrix.from_matrix([[int(L[i,j]) for j in range(L.ncols())] for i in range(L.nrows())])
    LLL.reduction(A)
    return matrix(ZZ, A.to_matrix())


In [ ]:
# Utilities 

from fpylll import *
from random import randrange
import numpy as np


"""
  Returns an n-dimensional vector,
  whose coordinates follow a centered binomial distribution
  with parameter eta.
"""
def binomial_vec(n, eta):
  v = np.array([0]*n)
  for i in range(n):
    v[i] = binomial_dist(2*eta) - eta
  return v

"""
  Returns an integer following the binomial distribution with parameter eta.
"""
def binomial_dist(eta):
  s = 0
  for i in range(eta):
    s += randrange(2)
  return s

"""
  Returns an n-dimensional vector,
  whose coordinates follow the uniform distrbution
  on [a,...,b-1].
"""
def uniform_vec(n, a, b):
  return np.array([randrange(a,b) for _ in range(n)])

"""
  Given a list of polynomials poly = [p_1, ...,p_n],
  this function returns an rows*cols matrix,
  consisting of the rotation matrices of p_1, ..., p_n.
"""
def module( polys, rows, cols ):
  if rows*cols != len(polys):
    raise ValueError("len(polys) has to equal rows*cols.")
  
  n = len(polys[0])
  for poly in polys:
    if len(poly) != n:
      raise ValueError("polys must not contain polynomials of varying degrees.")
  
  blocks = []
  
  for i in range(rows):
    row = []
    for j in range(cols):
      row.append( rotMatrix(polys[i*cols+j], cyclotomic=True) )
    blocks.append(row)
  
  return np.block( blocks )

def rotMatrix(poly, cyclotomic=False):
  n = len(poly)
  A = np.array( [[0]*n for _ in range(n)] )
  
  for i in range(n):
    for j in range(n):
      c = 1
      if cyclotomic and j < i:
        c = -1

      A[i][j] = c * poly[(j-i)%n]
      
  return A


In [ ]:
# LWE generation and hint generation 
# LWE generation adapted from https://github.com/juliannowakowski/lwe_with_hints

"""
  Returns A,s,e, as in Kyber.
"""
def kyberGen(variant):
  if variant not in [512,768,1024]:
    raise NotImplementedError("kyberGen(variant) supports only variant = 512, 768, 1024, but variant = %d was given." % variant)
  
  n = 256
  q = 3329
  k = variant//n
  
  if k == 2:
    eta = 3
  else:
    eta = 2
  
  s = binomial_vec(variant, eta)
  e = binomial_vec(variant, eta)
  
  polys = []
  for i in range(k*k):
    polys.append( uniform_vec(n,0,q) )
  
  A = module(polys, k, k)
  
  return A,s,e,q


def ntruGen(variant):
  if variant not in ["HPS-509", "HPS-677", "HPS-821", "HRSS"]:
    raise NotImplementedError("ntruGen(variant) supports only variant = HPS-509, HPS-677, HPS-821, HRSS but " + variant + " was given.")
    
  if variant == "HRSS":
    useHRSS = True
    n = 701
    q = 8192
  else:
    useHRSS = False
    n = int(variant[4:])
    if n == 821:
      q = 4096
    else:
      q = 2048
    
  generator = NTRUKeyGenerator(useHRSS, n, q)
  seed = generator.newSeed()
  f,g,h = generator.getKey(seed)
  
  A = rotMatrix(h)
  s = np.array(f)  
  e = -np.array(g)
  
  return A,s,e,q


def generateHints(s, q, k, centered):
  n = len(s)
  
  V = []
  L = []

  for i in range(k):
    if centered:
      v = np.array([ randrange( -int((q-1)/2), int((q+1)/2) ) for _ in range(n) ])
    else:
      v = np.array([ randrange(q) for _ in range(n) ])

    l = int(v.dot(s))
    
    V.append(v)
    L.append(l)
  
  return V,L

In [ ]:
def experiment(k,mm_LWE):
    import time
    A,s,e,q = kyberGen(512)
    #A,s,e,q = ntruGen("HPS-509")
    b = (s.dot(A) + e) % q
    # print(A)
    n = len(s)
    m = len(e)


    V,L = generateHints(s,q,k,centered='TRUE')
    # print(type(L))
    V = matrix(V).transpose()
    L = matrix(ZZ,vector(L))

    A = matrix(Zmod(q),A)
    e = matrix(vector(e))
    s = matrix(vector(s))
    b = matrix(vector(b))

    VV = block_matrix(ZZ,[[V,zero_matrix(n,1)],[(L),ones_matrix(1,1)]], subdivide=False)
    starttime = time.time()
    VV_HNF, UVV= VV.echelon_form(transformation=True)
    endtime = time.time()
    print('HNF time: %smin %ss'%(int(endtime-starttime)//60,(int(endtime-starttime)%60)))
    assert VV_HNF[k][-1] == 1

    print('HNF reduced!')
    print('-*'*20)


    SIS_0 = block_matrix(ZZ,[[block_matrix([[zero_matrix(k,n-k)],[identity_matrix(n-k)]]),zero_matrix(n,1)],[zero_matrix(1,n-k),ones_matrix(1,1)]], subdivide=False)
    
    SIS_1 = (UVV*SIS_0)[k:]

    print('first LLL reduce')
    print('-*'*20)

    SIS_B = matrix(SIS_1)
    SIS_B = flatter_reduce(SIS_B)
    print(SIS_B[0])
    print('target vector:',((s.transpose()[k:]).transpose()))
    if SIS_B[0].norm()*SIS_B[0].norm() <= ((s.transpose()[k:]).transpose()).norm()*((s.transpose()[k:]).transpose()).norm()+1+0.5:
        print('possible vectors:')
        for i in range(SIS_B.nrows() ):
            if SIS_B[i].norm()*SIS_B[i].norm() <= ((s.transpose()[k:]).transpose()).norm()*((s.transpose()[k:]).transpose()).norm()+1+0.5:
                print('row',i,':',SIS_B[i])
                print("LLL success!")
                return 1
    print('-*'*20)

    U_SIS_LLL = SIS_1.solve_left(SIS_B)

    print('-*'*20)

    UU = matrix(ZZ, U_SIS_LLL)

    mm0=0
    mm = 0
    print('mm:',mm)
    assert mm<m
    AB_part = matrix(ZZ,matrix(ZZ,U_SIS_LLL)*(UVV*block_matrix(Zmod(q),[[(A.transpose()[:mm]).transpose()],[(b.transpose()[:mm]).transpose()]], subdivide=False))[k:])

    for i in range(AB_part.nrows()):
        for j in range(AB_part.ncols()):
            if AB_part[i,j]>q//2:
                t = AB_part[i][j]
                AB_part[i,j] = t-q 

    SIS_LWE_basis = block_matrix([[q*identity_matrix(mm),zero_matrix(mm,n-k+1)],[AB_part,SIS_B]], subdivide=False)
    print('construct first SIS-LWE basis. mm:',mm)
    print('-*'*20)
    print('target:', (e.transpose()[:mm]).transpose(), (s.transpose()[k:]).transpose())



    from fpylll import BKZ as BKZ_FPYLLL, LLL, GSO, IntegerMatrix, FPLLL
    from fpylll.algorithms.bkz2 import BKZReduction
    FPLLL.set_precision(120)


    B = IntegerMatrix.from_matrix(SIS_LWE_basis)

    from fpylll.algorithms.bkz2 import BKZReduction as BKZ2
    MB = GSO.Mat(B, float_type="mpfr")
    MB.update_gso()
    bkz2 = BKZ2(MB)
    beta = 10
    maxBlocksize = 10
    foundsol = 0
    starttime = time.time()
    while foundsol==0  and  beta < maxBlocksize + 1:
        par = BKZ_FPYLLL.Param(beta, strategies=BKZ_FPYLLL.DEFAULT_STRATEGY, max_loops=8, flags=BKZ_FPYLLL.MAX_LOOPS)
        print('BKZ-',beta)
        bkz2(par)
        # print(MB.B[0])
        if MB.B[0].norm()*MB.B[0].norm() <= ((e.transpose()[:mm]).transpose()).norm()*((e.transpose()[:mm]).transpose()).norm() + ((s.transpose()[k:]).transpose()).norm()*((s.transpose()[k:]).transpose()).norm()+1+0.5:
            print('possible BKZ vectors:')
            for i in range(MB.B.nrows ):
                if MB.B[i].norm()*MB.B[i].norm() <= ((e.transpose()[:mm]).transpose()).norm()*((e.transpose()[:mm]).transpose()).norm() + ((s.transpose()[k:]).transpose()).norm()*((s.transpose()[k:]).transpose()).norm()+1+0.5:
                    foundsol = 1
                    print('SIS-BKZ success! beta:', beta)
                    endtime = time.time()
                    print('BKZ time: %smin %ss'%(int(endtime-starttime)//60,(int(endtime-starttime)%60)))
                    print(MB.B[i])
                    return 1
        beta+=1
    print('-*'*20)

    U_BKZ = matrix(ZZ,(SIS_LWE_basis.solve_left(matrix(B))).transpose()[mm-mm0:]).transpose()
    if mm0 > 0:
        U_BKZ = matrix(ZZ,(SIS_LWE_basis.solve_left(matrix(B))).transpose()[mm-mm0:]).transpose()
    UU = U_BKZ*UU
    mm0 = mm
    mm = mm_LWE
    print('construct new SIS-LWE basis. mm:',mm)
    assert mm<m
    assert mm0<=mm
    AB_part = matrix(ZZ,matrix(ZZ,UU)*(UVV*block_matrix(Zmod(q),[[(A.transpose()[:mm]).transpose()],[(b.transpose()[:mm]).transpose()]], subdivide=False))[k:])

    for i in range(AB_part.nrows()):
        for j in range(AB_part.ncols()):
            if AB_part[i,j]>q//2:
                t = AB_part[i][j]
                AB_part[i,j] = t-q 

    SIS_LWE_basis = block_matrix([[zero_matrix(mm-mm0,mm0),q*identity_matrix(mm-mm0),zero_matrix(mm-mm0,n-k+1)],[AB_part,UU*SIS_1]], subdivide=False)
    # print(SIS_LWE_basis)
    print('-*'*20)
    print('target vector:',(e.transpose()[:mm]).transpose(), (s.transpose()[k:]).transpose())

    from fpylll import BKZ as BKZ_FPYLLL, LLL, GSO, IntegerMatrix, FPLLL
    from fpylll.algorithms.bkz2 import BKZReduction
    import time

    FPLLL.set_precision(120)

    B = flatter_reduce(SIS_LWE_basis)

    print('target:',-(e.transpose()[:mm]).transpose(), (s.transpose()[k:]).transpose())
    print('-*'*20)
    print('after LLL')

    print(B[0])

    B = IntegerMatrix.from_matrix(B)

    from fpylll.algorithms.bkz2 import BKZReduction as BKZ2
    MB = GSO.Mat(B, float_type="mpfr")
    MB.update_gso()
    bkz2 = BKZ2(MB)
    beta = 5
    maxBlocksize = 40
    foundsol = 0
    starttime = time.time()
    while foundsol==0  and  beta < maxBlocksize + 1:
        par = BKZ_FPYLLL.Param(beta, strategies=BKZ_FPYLLL.DEFAULT_STRATEGY, max_loops=8, flags=BKZ_FPYLLL.MAX_LOOPS)
        print('BKZ-',beta)
        bkz2(par)
        # print(MB.B[0])
        if MB.B[0].norm()*MB.B[0].norm() <= ((e.transpose()[:mm]).transpose()).norm()*((e.transpose()[:mm]).transpose()).norm() + ((s.transpose()[k:]).transpose()).norm()*((s.transpose()[k:]).transpose()).norm()+1+0.5:
            endtime = time.time()
            print('BKZ time: %smin %ss'%(int(endtime-starttime)//60,(int(endtime-starttime)%60)))
            print('possible BKZ vectors:')
            for i in range(MB.B.nrows ):
                if MB.B[i].norm()*MB.B[i].norm() <= ((e.transpose()[:mm]).transpose()).norm()*((e.transpose()[:mm]).transpose()).norm() + ((s.transpose()[k:]).transpose()).norm()*((s.transpose()[k:]).transpose()).norm()+1+0.5:
                    foundsol = 1
                    print(MB.B[i])
                    return 1
        beta+=1
    print('-*'*20)

In [ ]:
startk=150
endk=200

for kk in range(startk,endk):
    print('k',startk+endk-kk,'number of LWE samples',10)
    for _ in range(50):
        experiment(startk+endk-kk, 10)
        print('####----'*20)